In [225]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix


# Data Cleaning and Preprocessing

### 1. Read the file using the correct header

In [226]:
train_original = pd.read_csv('pinkai_train.csv', header=1) 
test_original = pd.read_csv('pinkai_test.csv', header=1) 

### 2. Standardize column names

In [227]:
train_original.columns = train_original.columns.str.strip().str.lower().str.replace(' ', '_')
test_original.columns = test_original.columns.str.strip().str.lower().str.replace(' ', '_')

### Reading and Standardizing the Original Datasets

We loaded the original training and testing datasets using the correct header row and standardized the column names by removing extra spaces, converting them to lowercase, and replacing spaces with underscores.

This makes the column names consistent and easier to use during the data cleaning and analysis steps.


In [228]:
print(f"train: {train_original.shape}")
print(f"test: {test_original.shape}")

train: (9000, 38)
test: (3000, 37)


In [229]:
df_train = train_original.copy()
df_test = test_original.copy()

In [230]:
df_test.columns

Index(['patient_id', 'exam_date', 'center', 'age', 'biopsy_followup_code',
       'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
       'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave_points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'notes', 'sample_ref'],
      dtype='object')

In [231]:
df_train.columns

Index(['patient_id', 'exam_date', 'center', 'age', 'biopsy_followup_code',
       'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
       'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave_points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'notes', 'sample_ref',
       'diagnosis'],
      dtype='object')

### 3. Clean patient_id

In [232]:
df_test['patient_id']

0         912415
1       910309.0
2         911018
3         911255
4         910239
          ...   
2995    912193.0
2996      911534
2997      911815
2998      912057
2999      910147
Name: patient_id, Length: 3000, dtype: object

In [233]:
df_train['patient_id']

0          843421
1          846086
2          847672
3       ID-845611
4          849156
          ...    
8995       848308
8996       845034
8997     842812.0
8998       847060
8999       843724
Name: patient_id, Length: 9000, dtype: object

### Cleaning `patient_id`

The patient IDs came in three different shapes depending on how each record was entered — some plain (`912415`), some with a stray `.0` from being read as a float (`910309.0`), and some padded with commas or an `ID-` prefix. We strip all of that noise down to a bare number, then cast to `Int64` so missing IDs can coexist with valid integers without breaking the column's type.

In [234]:
for df in [df_train, df_test]:
    df['patient_id'] = (
        df['patient_id']
        .astype(str)
        .str.strip()
        .str.replace('ID-', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('.0', '', regex=False)
    )

    df['patient_id'] = pd.to_numeric(
        df['patient_id'],
        errors='coerce'
    ).astype('Int64')

In [235]:
df_train['patient_id'].head()

0    843421
1    846086
2    847672
3    845611
4    849156
Name: patient_id, dtype: Int64

### Converting measurement columns from text to numbers

All the measurement columns (`_mean`, `_se`, `_worst`) were read in as text, mainly because some values use a comma as the decimal separator (`"77,6045"` instead of `77.6045`) — a European-style number format mixed into what's otherwise a standard CSV. We swap the comma for a period and coerce to numeric, which also turns any leftover junk values (`"missing"`, `"unknown"`, empty strings) into clean `NaN`s automatically, since they can't be parsed as numbers either way.

In [236]:
for df in [df_train, df_test]:
    for col in df.select_dtypes(include='object').columns:
        if col not in [
            'exam_date',
            'center',
            'notes',
            'sample_ref',
            'biopsy_followup_code',
            'diagnosis'
        ]:
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')

### 5. Convert exam_date to datetime
Dates were entered by hand across multiple source systems, so they show up in different formats — some ISO (`2026-09-05`), some spelled out (`"June 28, 2025"`). `format='mixed'` lets pandas infer the format row by row instead of forcing one pattern on everything, and `dayfirst=True` resolves ambiguous numeric dates (like `03/04/2025`) as day-before-month. Anything that still can't be parsed becomes `NaT` rather than crashing the whole column.

In [237]:
for df in [df_train, df_test]:
    df['exam_date'] = pd.to_datetime(
        df['exam_date'],
        errors='coerce',
        format='mixed',
        dayfirst=True
    )

In [238]:
df_test.head()

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,notes,sample_ref
0,912415,2026-09-05,Site B,62,ONC-REF,11.6775,14.7006,77.6045,426.6,0.080439,0.044300,0.008406,0.010932,0.154470,0.061130,0.235100,0.990342,1.42900,15.8,0.004453,0.009787,0.006468,0.559475,0.017194,0.001898,12.5557,21.7473,84.9603,482.0,0.110190,NaN,0.044077,0.053367,NaN,0.070833,NaN,REF-6579-B
1,910309,2025-06-28,SITE B,70,ROUTINE,15.6750,20.6795,107.9500,NaN,0.094289,0.107306,0.157221,NaN,0.179496,0.059951,0.335507,0.746803,2.20470,31.6,0.006189,0.036137,0.049671,NaN,0.021874,0.005125,16.8502,24.5066,108.0100,873.6,0.127763,0.252091,0.464187,0.120361,0.311598,0.093128,NaN,REF-8343-H
2,911018,2026-01-19,Site A,59,ONC-REF,18.0691,20.7000,116.2800,1028.2,0.122007,0.198219,0.200772,0.118084,0.219194,0.068350,0.969436,2.112790,6.97736,88.4,0.009790,0.071537,0.055064,0.020561,0.067099,0.005721,19.3605,24.8327,129.4500,1098.9,0.146008,0.299124,0.269405,0.173723,0.352651,0.077287,prior reading 0.114,REF-9464-G
3,911255,2026-03-28,Site B,52,ROUTINE,14.5711,14.9292,92.8657,641.6,0.100500,0.129625,0.108261,0.068581,0.173500,0.070654,0.546189,1.670610,4.08834,47.0,0.004130,0.046386,0.054077,0.020287,0.028386,0.007756,16.3583,18.6306,104.9300,794.3,0.102602,0.198113,0.212689,0.120160,0.244305,NaN,NaN,REF-2782-A
4,910239,2025-04-16,Site A,81,ROUTINE,18.8593,16.1591,126.9500,NaN,0.087628,0.083328,0.100857,0.069174,NaN,0.051971,0.383681,0.731810,2.40321,46.9,0.002859,0.007861,0.015503,0.007215,0.010433,0.000592,22.6268,25.0624,145.6000,NaN,0.120501,0.200687,0.295089,0.157310,0.333516,0.068490,"measured 18,3 on intake",REF-3472-D


### 6. Standardize Center Values

In [239]:
df_train['center'].value_counts()

center
Site A       3308
Site B       2835
Site C       1977
SITE A        166
SITE B        149
SITE C         85
 Site B        36
 Site A        35
  Site C       32
  Site A       31
  Site B       31
  Site A       30
 Site A        29
  Site B       28
\tSite A       25
\tSite B       24
 Site B        23
\tSite A       23
\tSite B       20
\tSite C       20
  Site C       19
\tSite C       19
 Site C        18
 Site C        16
 SITE C         3
  SITE B        2
  SITE C        2
 SITE A         2
 SITE C         2
\tSITE A        2
\tSITE C        1
\tSITE A        1
  SITE A        1
\tSITE C        1
\tSITE B        1
  SITE A        1
 SITE B         1
 SITE A         1
Name: count, dtype: int64

### Standardizing `center`

Since each clinic entered records independently, `center` shows up with mixed casing — `"SITE B"`, `"site b"`, `"Site B"` — all meaning the same thing. Stripping whitespace and applying `.str.title()` collapses these down to a consistent `"Site B"` style across the board.

In [240]:
for df in [df_train, df_test]:
    df['center'] = (
        df['center']
        .astype(str)
        .str.strip()
        .str.title()
    )

In [241]:
df_train['center'].value_counts()

center
Site A    3655
Site B    3150
Site C    2195
Name: count, dtype: int64

In [242]:
df_test['center'].value_counts()

center
Site A    1161
Site B    1116
Site C     723
Name: count, dtype: int64

In [243]:
df_test['biopsy_followup_code'].value_counts()

biopsy_followup_code
ROUTINE    2300
ONC-REF     625
RECHECK      75
Name: count, dtype: int64

### Dropping `notes`

`notes` is free text, typed manually, and missing in nearly half the rows (~48% in train). It's unstructured and doesn't fit a standard tabular model without NLP work we don't have time for in this format, so we drop it rather than try to extract signal from it.

In [244]:
df_train.drop(columns=['notes'], inplace=True)
df_test.drop(columns=['notes'], inplace=True)

In [245]:
df_test['sample_ref'].value_counts()

sample_ref
REF-8209-C    3
REF-6786-G    3
REF-5524-E    3
REF-9581-D    3
REF-8675-B    3
             ..
REF-3087-H    1
REF-6480-E    1
REF-3544-E    1
REF-3804-H    1
REF-9817-E    1
Name: count, Length: 2826, dtype: int64

### Dropping `sample_ref`

Confirmed unique per row — it's the lab's own internal tracking string for the slide, not a property of the tumor itself. No predictive value, so we drop it as a junk identifier column.

In [246]:
df_train.drop(columns=['sample_ref'], inplace=True)
df_test.drop(columns=['sample_ref'], inplace=True)

In [247]:
df_train['diagnosis'].value_counts() 
# 0 = Benign
# 1 = Malignant

diagnosis
benign          1078
Benign          1074
 B              1066
0               1020
B                979
                ... 
\tMALIGNANT        4
\tmalignant        4
  M                3
  1                2
 malignant         1
Name: count, Length: 67, dtype: int64

### Encoding `diagnosis` as 0/1

The target column had ~67 inconsistent variants for just two real categories — mixed case, numeric codes, stray whitespace, even tab characters. We strip and lowercase everything first so `"Benign"`, `" B"`, `"\tmalignant"`, and `"1"` all collapse to the same handful of forms, then map those onto a clean binary: `0` for benign, `1` for malignant.

In [248]:
df_train['diagnosis'] = (
    df_train['diagnosis']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'b': 0,
        '0': 0,
        'benign': 0,
        'm': 1,
        '1': 1,
        'malignant': 1
    })
)

In [249]:
df_train['diagnosis'].value_counts() 


diagnosis
0    5494
1    3506
Name: count, dtype: int64

In [250]:
df_test.head(30)

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,912415,2026-09-05,Site B,62,ONC-REF,11.67750,14.7006,77.6045,426.6,0.080439,0.044300,8.406040e-03,1.093160e-02,0.154470,0.061130,0.235100,0.990342,1.429000,15.8,0.004453,0.009787,0.006468,5.594750e-01,0.017194,0.001898,12.55570,21.7473,84.9603,482.0,0.110190,NaN,4.407670e-02,0.053367,NaN,0.070833
1,910309,2025-06-28,Site B,70,ROUTINE,15.67500,20.6795,107.9500,NaN,0.094289,0.107306,1.572210e-01,NaN,0.179496,0.059951,0.335507,0.746803,2.204700,31.6,0.006189,0.036137,0.049671,NaN,0.021874,0.005125,16.85020,24.5066,108.0100,873.6,0.127763,0.252091,4.641870e-01,0.120361,0.311598,0.093128
2,911018,2026-01-19,Site A,59,ONC-REF,18.06910,20.7000,116.2800,1028.2,0.122007,0.198219,2.007720e-01,1.180840e-01,0.219194,0.068350,0.969436,2.112790,6.977360,88.4,0.009790,0.071537,0.055064,2.056080e-02,0.067099,0.005721,19.36050,24.8327,129.4500,1098.9,0.146008,0.299124,2.694050e-01,0.173723,0.352651,0.077287
3,911255,2026-03-28,Site B,52,ROUTINE,14.57110,14.9292,92.8657,641.6,0.100500,0.129625,1.082610e-01,6.858060e-02,0.173500,0.070654,0.546189,1.670610,4.088340,47.0,0.004130,0.046386,0.054077,2.028750e-02,0.028386,0.007756,16.35830,18.6306,104.9300,794.3,0.102602,0.198113,2.126890e-01,0.120160,0.244305,NaN
4,910239,2025-04-16,Site A,81,ROUTINE,18.85930,16.1591,126.9500,NaN,0.087628,0.083328,1.008570e-01,6.917390e-02,NaN,0.051971,0.383681,0.731810,2.403210,46.9,0.002859,0.007861,0.015503,7.215150e-03,0.010433,0.000592,22.62680,25.0624,145.6000,NaN,0.120501,0.200687,2.950890e-01,0.157310,0.333516,0.068490
5,912582,2025-11-09,Site B,75,ONC-REF,9.45810,16.4233,60.1500,277.6,0.079674,0.068951,2.354550e-02,8.379400e-12,0.172600,0.065462,0.113857,0.495743,0.768977,6.7,0.006736,0.021365,0.026147,3.388510e-03,0.018697,0.003855,10.75770,22.1384,71.7108,354.8,0.133422,0.236329,2.647460e-01,0.055461,0.351149,0.099686
6,911291,2025-02-13,Site A,63,ONC-REF,15.03100,16.8732,98.4084,695.3,0.095587,0.103835,9.185130e-02,5.741360e-02,0.156412,0.058797,0.298536,0.945507,2.356500,26.2,0.005167,0.032112,0.046127,1.390650e-02,0.010563,0.003586,17.28500,21.2621,109.6200,917.3,0.121124,0.290864,3.745200e-01,0.126120,0.186861,0.080323
7,911627,2025-03-18,Site C,69,ROUTINE,12.54550,29.5450,83.8451,478.8,0.100700,0.076307,4.483940e-02,7.690900e-03,0.180452,0.063728,0.343090,2.046230,2.518690,16.5,0.008539,0.027398,0.067211,1.382660e-02,0.025719,0.004236,12.55810,38.2755,83.8870,479.0,0.123400,0.157405,1.375260e-01,0.064117,0.261841,0.073971
8,910635,2025-10-20,Site C,65,ROUTINE,1.87039,16.1700,121.6500,NaN,0.092339,0.084982,1.332340e-01,3.484170e-02,0.208772,0.058539,0.262476,0.675737,2.138930,26.4,0.005761,0.019091,0.048772,1.252760e-02,0.019722,0.003407,18.72260,19.9536,121.7100,1078.5,0.121868,0.218678,2.388820e-01,0.107999,0.339718,0.077982
9,911617,2026-02-03,Site C,29,RECHECK,8.85126,17.0574,57.5912,253.7,0.079666,0.058195,4.922280e-02,2.304360e-02,0.156751,0.063442,0.300355,1.331200,1.559980,20.2,0.005969,0.017033,0.034667,7.604450e-03,0.015208,0.002483,12.12290,25.5000,94.1669,459.3,0.119733,0.188414,2.280770e-01,0.080760,0.256109,0.080146


### Re-checking both datasets after cleaning

Same inspection as the very first step, run again now that `patient_id`, the measurement columns, `exam_date`, `center`, and `diagnosis` have all been cleaned. This is the checkpoint that confirms the fixes actually worked — dtypes should now be numeric where expected, and `diagnosis` should show just two clean values instead of 67.

In [251]:
# ==============================
# Train Dataset - Initial Inspection
# ==============================

print("Shape:")
print(df_train.shape)

print("\nData Types:")
print(df_train.dtypes)

print("\nMissing Values:")
print(df_train.isnull().sum())

print("\nDuplicate Rows:")
print(df_train.duplicated().sum())

print("\nDiagnosis Distribution:")
print(df_train['diagnosis'].value_counts())


# ==============================
# Test Dataset - Initial Inspection
# ==============================

print("\n" + "="*40)

print("Shape:")
print(df_test.shape)

print("\nData Types:")
print(df_test.dtypes)

print("\nMissing Values:")
print(df_test.isnull().sum())

print("\nDuplicate Rows:")
print(df_test.duplicated().sum())

Shape:
(9000, 36)

Data Types:
patient_id                          Int64
exam_date                  datetime64[ns]
center                             object
age                                 int64
biopsy_followup_code               object
radius_mean                       float64
texture_mean                      float64
perimeter_mean                    float64
area_mean                         float64
smoothness_mean                   float64
compactness_mean                  float64
concavity_mean                    float64
concave_points_mean               float64
symmetry_mean                     float64
fractal_dimension_mean            float64
radius_se                         float64
texture_se                        float64
perimeter_se                      float64
area_se                           float64
smoothness_se                     float64
compactness_se                    float64
concavity_se                      float64
concave_points_se                 float64
sym

## Initial EDA Findings

### Train Dataset

* The dataset contains **9,000 rows and 38 columns**.
* There are **173 duplicate rows**.
* Several columns contain missing values.
* The `notes` column has **4,297 missing values**, which is a large proportion of the dataset.
* Most medical numerical features are stored as `object` instead of numeric types and will need conversion.
* The `exam_date` column is also stored as `object` and will need to be converted to a datetime format.
* The `diagnosis` column contains **67 different values**, mainly due to inconsistent representations such as:

  * `benign`, `Benign`, `B`, `0`
  * `malignant`, `M`, `1`

### Test Dataset

* The dataset contains **3,000 rows and 37 columns**.
* There are **58 duplicate rows**.
* Several columns contain missing values.
* The `notes` column has **1,448 missing values**.
* Most medical numerical features are stored as `object` and will need conversion.
* The `exam_date` column is also stored as `object`.

### Main Data Quality Issues

The initial EDA identified several issues that need to be addressed during data cleaning:

1. Duplicate rows.
2. Missing values.
3. Numerical features stored as `object`.
4. Dates stored as `object`.
5. Inconsistent `diagnosis` labels.
6. A large number of missing values in the `notes` column.

These issues will be investigated further during the cleaning and preprocessing stages.


In [252]:
df_test.head()

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,912415,2026-09-05,Site B,62,ONC-REF,11.6775,14.7006,77.6045,426.6,0.080439,0.044300,0.008406,0.010932,0.154470,0.061130,0.235100,0.990342,1.42900,15.8,0.004453,0.009787,0.006468,0.559475,0.017194,0.001898,12.5557,21.7473,84.9603,482.0,0.110190,NaN,0.044077,0.053367,NaN,0.070833
1,910309,2025-06-28,Site B,70,ROUTINE,15.6750,20.6795,107.9500,NaN,0.094289,0.107306,0.157221,NaN,0.179496,0.059951,0.335507,0.746803,2.20470,31.6,0.006189,0.036137,0.049671,NaN,0.021874,0.005125,16.8502,24.5066,108.0100,873.6,0.127763,0.252091,0.464187,0.120361,0.311598,0.093128
2,911018,2026-01-19,Site A,59,ONC-REF,18.0691,20.7000,116.2800,1028.2,0.122007,0.198219,0.200772,0.118084,0.219194,0.068350,0.969436,2.112790,6.97736,88.4,0.009790,0.071537,0.055064,0.020561,0.067099,0.005721,19.3605,24.8327,129.4500,1098.9,0.146008,0.299124,0.269405,0.173723,0.352651,0.077287
3,911255,2026-03-28,Site B,52,ROUTINE,14.5711,14.9292,92.8657,641.6,0.100500,0.129625,0.108261,0.068581,0.173500,0.070654,0.546189,1.670610,4.08834,47.0,0.004130,0.046386,0.054077,0.020287,0.028386,0.007756,16.3583,18.6306,104.9300,794.3,0.102602,0.198113,0.212689,0.120160,0.244305,NaN
4,910239,2025-04-16,Site A,81,ROUTINE,18.8593,16.1591,126.9500,NaN,0.087628,0.083328,0.100857,0.069174,NaN,0.051971,0.383681,0.731810,2.40321,46.9,0.002859,0.007861,0.015503,0.007215,0.010433,0.000592,22.6268,25.0624,145.6000,NaN,0.120501,0.200687,0.295089,0.157310,0.333516,0.068490


In [253]:
df_train[df_train.duplicated(keep=False)].sort_values(
    by='patient_id'
).head(20)

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
375,842059,2025-02-26,Site A,50,ROUTINE,12.06000,13.4090,78.6871,440.7,0.103209,0.057354,0.040698,0.019691,0.180726,0.063164,0.201810,1.110580,NaN,12.4,0.007760,0.008743,0.019026,0.009153,0.027881,0.002242,12.31820,17.0512,80.6362,449.8,0.126400,0.070582,0.079601,0.040453,0.290055,0.064407,0
4196,842059,2025-02-26,Site A,50,ROUTINE,12.06000,13.4090,78.6871,440.7,0.103209,0.057354,0.040698,0.019691,0.180726,0.063164,0.201810,1.110580,NaN,12.4,0.007760,0.008743,0.019026,0.009153,0.027881,0.002242,12.31820,17.0512,80.6362,449.8,0.126400,0.070582,0.079601,0.040453,0.290055,0.064407,0
8987,842061,2026-05-17,Site B,56,ROUTINE,13.27000,15.4550,82.2851,540.2,0.074451,0.045531,0.027114,0.012461,0.156807,0.056690,0.200480,0.779579,1.46381,16.9,0.005267,0.013700,0.026129,0.008401,0.018174,0.002378,14.03860,19.6838,91.6761,557.5,0.098173,0.125459,0.188905,0.063193,0.253465,0.067447,0
4453,842061,2026-05-17,Site B,56,ROUTINE,13.27000,15.4550,82.2851,540.2,0.074451,0.045531,0.027114,0.012461,0.156807,0.056690,0.200480,0.779579,1.46381,16.9,0.005267,0.013700,0.026129,0.008401,0.018174,0.002378,14.03860,19.6838,91.6761,557.5,0.098173,0.125459,0.188905,0.063193,0.253465,0.067447,0
2952,842078,2025-12-21,Site C,67,ONC-REF,18.64940,NaN,116.5700,NaN,0.093857,0.148780,0.234625,0.091727,0.183352,0.055888,0.456306,1.316690,3.09117,128.3,NaN,0.038563,NaN,0.013941,0.014871,0.003224,18.66800,38.0330,116.6300,1070.6,0.138900,0.373500,0.591185,0.183789,0.281218,0.079147,1
1648,842078,2025-12-21,Site C,67,ONC-REF,18.64940,NaN,116.5700,NaN,0.093857,0.148780,0.234625,0.091727,0.183352,0.055888,0.456306,1.316690,3.09117,128.3,NaN,0.038563,NaN,0.013941,0.014871,0.003224,18.66800,38.0330,116.6300,1070.6,0.138900,0.373500,0.591185,0.183789,0.281218,0.079147,1
3928,842102,2026-08-25,Site A,37,ROUTINE,12.39250,NaN,83.4398,468.7,0.113304,0.174745,0.091564,0.054254,0.200139,0.072524,0.460933,2.500710,3.70024,32.7,0.010983,0.059063,0.020196,0.015059,0.029143,0.006771,13.65100,37.1306,88.9666,553.0,0.142700,0.362109,0.181100,0.082747,0.255284,0.104149,0
2771,842102,2026-08-25,Site A,37,ROUTINE,12.39250,NaN,83.4398,468.7,0.113304,0.174745,0.091564,0.054254,0.200139,0.072524,0.460933,2.500710,3.70024,32.7,0.010983,0.059063,0.020196,0.015059,0.029143,0.006771,13.65100,37.1306,88.9666,553.0,0.142700,0.362109,0.181100,0.082747,0.255284,0.104149,0
4243,842131,2025-01-06,Site A,53,ONC-REF,11.50190,26.5551,76.6936,396.8,0.110900,0.155615,0.153191,0.065262,0.192659,NaN,0.239063,2.163250,1.91156,19.1,0.007160,0.061732,NaN,0.015051,0.030941,0.005860,13.44270,41.7648,88.3725,545.0,0.149217,0.584824,0.502298,0.149181,0.364545,0.111487,1
8075,842131,2025-01-06,Site A,53,ONC-REF,11.50190,26.5551,76.6936,396.8,0.110900,0.155615,0.153191,0.065262,0.192659,NaN,0.239063,2.163250,1.91156,19.1,0.007160,0.061732,NaN,0.015051,0.030941,0.005860,13.44270,41.7648,88.3725,545.0,0.149217,0.584824,0.502298,0.149181,0.364545,0.111487,1


In [254]:
df_test[df_test.duplicated(keep=False)].sort_values(
    by='patient_id'
).head(20)

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
205,910008,2026-09-12,Site A,70,ROUTINE,13.04830,20.7067,84.0513,502.7,0.082047,0.076805,0.086501,0.030230,0.161860,0.058311,0.229479,1.501580,1.83986,20.8,0.006874,0.024879,0.048436,0.011134,0.025756,NaN,14.0079,28.8361,88.4337,600.9,0.107193,0.235256,0.312078,0.089994,0.300226,0.071276
1547,910008,2026-09-12,Site A,70,ROUTINE,13.04830,20.7067,84.0513,502.7,0.082047,0.076805,0.086501,0.030230,0.161860,0.058311,0.229479,1.501580,1.83986,20.8,0.006874,0.024879,0.048436,0.011134,0.025756,NaN,14.0079,28.8361,88.4337,600.9,0.107193,0.235256,0.312078,0.089994,0.300226,0.071276
1592,910165,2026-01-01,Site A,61,ROUTINE,15.22530,18.1785,713.6200,99.9,0.091054,0.057727,0.039147,NaN,0.166523,0.054176,0.260406,0.779969,1.54042,26.4,0.005678,0.010942,0.018063,0.009474,0.016730,0.001891,16.6573,22.2213,110.3600,839.8,0.124708,0.057727,0.039147,0.071275,0.236975,0.058162
2218,910165,2026-01-01,Site A,61,ROUTINE,15.22530,18.1785,713.6200,99.9,0.091054,0.057727,0.039147,NaN,0.166523,0.054176,0.260406,0.779969,1.54042,26.4,0.005678,0.010942,0.018063,0.009474,0.016730,0.001891,16.6573,22.2213,110.3600,839.8,0.124708,0.057727,0.039147,0.071275,0.236975,0.058162
1709,910206,2025-08-09,Site C,57,ROUTINE,9.43286,NaN,63.5833,279.1,0.084749,0.059297,0.059572,0.013608,0.173836,0.061055,0.173768,0.781126,1.52774,-441.3,0.008873,0.020020,0.034953,0.010323,0.021296,0.003419,10.9760,19.7464,72.9616,370.1,NaN,0.122548,0.145262,0.059235,0.273838,0.074190
801,910206,2025-08-09,Site C,57,ROUTINE,9.43286,NaN,63.5833,279.1,0.084749,0.059297,0.059572,0.013608,0.173836,0.061055,0.173768,0.781126,1.52774,-441.3,0.008873,0.020020,0.034953,0.010323,0.021296,0.003419,10.9760,19.7464,72.9616,370.1,NaN,0.122548,0.145262,0.059235,0.273838,0.074190
733,910242,2026-02-08,Site A,35,ROUTINE,14.40650,15.9017,91.2513,658.8,0.098807,0.084034,0.067260,0.062317,0.135895,0.058214,0.211677,1.202070,NaN,21.0,0.006690,0.012525,0.014581,0.012860,0.013154,0.002069,16.5873,23.2044,107.6300,847.2,0.141300,0.182303,0.164962,0.144880,0.253185,0.075922
1631,910242,2026-02-08,Site A,35,ROUTINE,14.40650,15.9017,91.2513,658.8,0.098807,0.084034,0.067260,0.062317,0.135895,0.058214,0.211677,1.202070,NaN,21.0,0.006690,0.012525,0.014581,0.012860,0.013154,0.002069,16.5873,23.2044,107.6300,847.2,0.141300,0.182303,0.164962,0.144880,0.253185,0.075922
981,910317,2025-07-26,Site C,43,ROUTINE,1.09872,27.1133,69.4096,363.5,0.089885,0.066927,0.105818,0.022869,0.180248,0.063008,0.299278,2.605490,1.95400,37.7,0.008536,0.012253,0.000000,0.008179,0.026771,0.002218,10.9982,35.8141,72.2431,383.2,0.110316,NaN,NaN,0.025640,0.270385,0.066704
1502,910317,2025-07-26,Site C,43,ROUTINE,1.09872,27.1133,69.4096,363.5,0.089885,0.066927,0.105818,0.022869,0.180248,0.063008,0.299278,2.605490,1.95400,37.7,0.008536,0.012253,0.000000,0.008179,0.026771,0.002218,10.9982,35.8141,72.2431,383.2,0.110316,NaN,NaN,0.025640,0.270385,0.066704


### Dropping duplicate rows

Doing this *after* cleaning (not before) matters — before cleaning, two identical records could look different on paper (one with `"77,6045"`, the other with `77.6045`) and `duplicated()` would miss them. Now that formatting is normalized, an exact match really means the same record twice. We drop and reset the index so row numbers stay contiguous for later steps like `GroupKFold`.

In [255]:
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_test = df_test.drop_duplicates().reset_index(drop=True)

### Checking patient ID consistency

Two things here. First, how many rows share a `patient_id` — that tells us how many patients were sampled more than once, which matters for grouping in cross-validation. Second, and more serious: whether any single patient has *conflicting* diagnoses recorded (both benign and malignant) — that would be a genuine data integrity problem, not just a formatting one, and worth flagging regardless of what we do with it.

In [273]:
# ==============================
# Check Patient IDs
# ==============================

print("Repeated patient IDs in train:")
print(df_train['patient_id'].duplicated().sum())

print("\nRepeated patient IDs in test:")
print(df_test['patient_id'].duplicated().sum())


# ==============================
# Check Diagnosis Consistency
# ==============================

print("\nPatients with different diagnoses:")
print(
    df_train.groupby('patient_id')['diagnosis']
    .nunique()
    .value_counts()
)

Repeated patient IDs in train:
173

Repeated patient IDs in test:
58

Patients with different diagnoses:
diagnosis
1    8604
2      50
Name: count, dtype: int64


### Inspecting conflicting diagnoses

Pulling up the actual rows for any patient flagged with more than one unique `diagnosis` value, sorted by `patient_id` so each patient's records sit next to each other for a direct comparison — including `exam_date`, to check whether the conflict could be a genuine change over time rather than a data entry error.

In [257]:
conflicting_patients = (
    df_train.groupby('patient_id')['diagnosis']
    .nunique()
)

conflicting_patients = conflicting_patients[
    conflicting_patients > 1
].index

df_train[
    df_train['patient_id'].isin(conflicting_patients)
].sort_values('patient_id').head(20)

,patient_id,exam_date,center,age,biopsy_followup_code,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
1758,842040,2025-09-27,Site B,66,ROUTINE,13.53090,23.3143,86.5699,550.7,0.089496,0.099045,0.060014,3.715080e-02,0.158831,0.064204,0.181153,0.922171,1.16496,14.7,0.004220,0.013423,NaN,0.009922,0.014656,0.002787,14.89570,34.2344,100.6000,684.9,NaN,0.216162,0.230189,1.179890e-01,0.272222,0.086381,0
8292,842040,2025-09-27,Site B,66,ROUTINE,13.52920,23.4023,85.9442,550.9,0.089031,0.099041,0.060117,3.733440e-02,0.159102,0.064110,0.181499,0.927780,1.16699,14.7,0.004188,0.013433,NaN,0.009957,0.014635,0.002787,14.87320,34.1286,100.6700,682.2,NaN,0.216828,0.230065,1.178670e-01,0.272128,0.086223,1
8255,842424,2026-06-07,Site A,72,ONC-REF,14.21190,20.6667,92.1895,633.3,0.092757,0.127418,0.098743,5.837470e-02,NaN,0.064116,0.341605,1.149060,2.18311,31.6,0.004895,0.031137,0.034726,0.012689,0.026348,0.004370,17.38230,27.6496,125.9200,964.5,0.113914,NaN,0.321494,1.611000e-01,0.400881,9.869960,1
4257,842424,2026-06-07,Site A,72,ONC-REF,14.14280,20.6255,92.0418,632.7,0.093152,0.127956,0.098808,5.861110e-02,NaN,0.064390,0.340906,1.148570,2.18350,31.6,0.004877,0.030993,0.034538,0.012729,0.026267,0.004377,17.35810,27.5928,125.6300,964.9,0.114371,NaN,0.322966,1.615740e-01,0.400495,9.873260,0
7673,842426,2025-01-23,Site B,50,ROUTINE,11.31320,14.0373,73.7379,392.2,0.117098,0.115498,0.085894,4.810060e-02,0.208785,0.076810,0.430425,1.491440,3.01816,27.6,0.005366,0.034959,0.051046,0.016353,0.022977,0.007607,NaN,16.6414,75.7537,437.5,0.120732,0.115498,0.085894,5.449790e-02,0.240400,0.082840,0
5423,842426,2025-01-23,Site B,50,ROUTINE,11.28070,14.0579,73.7686,390.5,0.116986,0.116023,0.085695,4.816290e-02,0.208560,0.076571,0.430631,1.493760,3.02118,27.7,0.005386,0.034988,0.050875,0.016400,0.023027,0.007616,NaN,16.7376,75.7832,437.4,0.120690,0.115682,0.086354,5.431000e-02,0.240985,0.083359,1
5324,842522,2026-08-01,Site A,54,ROUTINE,14.80260,22.2200,94.9879,663.8,0.079457,0.096594,0.059292,2.788120e-02,0.173500,0.059934,0.281714,1.490030,2.23915,23.3,0.004479,0.020073,0.021631,0.007962,0.018421,0.004588,16.46710,35.7416,103.8000,839.5,0.104899,0.227390,0.203976,1.048010e-01,0.287100,0.083181,0
6495,842522,2026-08-01,Site A,54,ROUTINE,14.76780,22.2081,94.5845,662.1,0.079850,0.096811,0.059224,2.800110e-02,0.173327,0.059983,0.283014,1.489840,2.23757,23.2,0.004485,0.020062,0.021546,0.007973,0.018309,0.004580,16.52090,35.5033,103.8800,837.2,0.104944,0.227972,0.203539,1.050440e-01,0.288213,0.083140,1
7742,842538,2025-05-17,Site B,56,ONC-REF,17.28070,18.4582,NaN,938.0,0.090575,0.129405,0.141266,5.575960e-02,0.209922,0.061657,0.419076,0.775258,NaN,45.1,0.004031,0.024971,0.030575,0.007847,0.018455,0.004098,21.47480,2657.6700,140.1200,1444.8,0.135989,0.415800,NaN,1.545430e-01,0.436244,0.108464,0
6582,842538,2025-05-17,Site B,56,ONC-REF,17.28990,18.4812,NaN,932.6,0.090560,0.129639,0.141677,5.609850e-02,0.209187,0.061454,0.420792,0.779931,NaN,44.9,0.004021,0.024931,0.030507,0.007809,0.018413,0.004104,21.55460,2656.6300,140.0100,NaN,0.136385,0.418491,NaN,1.541530e-01,0.437058,0.108652,1


### Missing value summary (post-cleaning)

A clean, sorted view of exactly which columns still have gaps and how bad each one is, now that the disguised missing markers (`"missing"`, `"unknown"`) have been converted to real `NaN`s during the numeric conversion step. This is the real missing-data picture — the earlier `isnull().sum()` before cleaning understated it.

In [258]:
missing_train = (
    df_train.isnull().sum()
    .to_frame('missing_count')
)

missing_train['missing_percent'] = (
    missing_train['missing_count']
    / len(df_train) * 100
)

missing_train = missing_train[
    missing_train['missing_count'] > 0
].sort_values(
    'missing_percent',
    ascending=False
)

missing_train

,missing_count,missing_percent
area_worst,1318,14.931460
area_mean,934,10.581171
smoothness_se,696,7.884899
smoothness_worst,693,7.850912
smoothness_mean,687,7.782939
fractal_dimension_mean,483,5.471848
concave_points_worst,467,5.290586
compactness_worst,458,5.188626
concavity_worst,457,5.177297
perimeter_worst,455,5.154639


In [259]:
print(df_train['biopsy_followup_code'].value_counts(dropna=False))

biopsy_followup_code
ROUTINE    5255
ONC-REF    3387
RECHECK     185
Name: count, dtype: int64


### Relationship Between Biopsy Follow-Up Code and Diagnosis

The following cross-tabulation examines the relationship between `biopsy_followup_code` and `diagnosis`.

The `normalize='index'` parameter converts the counts into proportions within each `biopsy_followup_code` category. This allows us to compare the distribution of diagnoses across different biopsy follow-up groups.

In [261]:
pd.crosstab(
    df_train['biopsy_followup_code'],
    df_train['diagnosis'],
    normalize='index'
)

diagnosis,0,1
biopsy_followup_code,,
ONC-REF,0.039268,0.960732
RECHECK,0.583784,0.416216
ROUTINE,0.979068,0.020932


### Diagnosis Distribution

The histogram shows the distribution of the target variable, `diagnosis`, in the training dataset.

The diagnosis values are:
- **0 = Benign**
- **1 = Malignant**

The number of observations in each category helps us understand whether the dataset is balanced or if there is a class imbalance between benign and malignant cases.

In [262]:
import plotly.express as px

fig = px.histogram(
    df_train,
    x='diagnosis',
    color='diagnosis',
    title='Diagnosis Distribution',
    labels={
        'diagnosis': 'Diagnosis (0 = Benign, 1 = Malignant)',
        'count': 'Count'
    },
    text_auto=True
)

fig.show()

### Feature Distribution by Diagnosis

The following box plots compare the distributions of selected tumor features between benign and malignant diagnoses.

The features analyzed are:
- `radius_mean`
- `texture_mean`
- `perimeter_mean`
- `area_mean`

Box plots are useful for comparing the **median, spread, and potential outliers** of each feature across the two diagnosis groups. This helps identify whether these features show noticeable differences between benign and malignant cases.

In [263]:
import plotly.express as px

features = [
    'radius_mean',
    'texture_mean',
    'perimeter_mean',
    'area_mean'
]

for feature in features:
    fig = px.box(
        df_train,
        x='diagnosis',
        y=feature,
        color='diagnosis',
        title=f'{feature} vs Diagnosis',
        labels={
            'diagnosis': 'Diagnosis (0 = Benign, 1 = Malignant)',
            feature: feature.replace('_', ' ').title()
        }
    )

    fig.show()

### Feature Correlation with Diagnosis

This analysis calculates the correlation between each numeric feature and the `diagnosis` variable.

The correlation coefficient measures the **strength and direction of the linear relationship** between a feature and diagnosis:

- **Positive correlation:** Higher feature values tend to be associated with higher diagnosis values.
- **Negative correlation:** Higher feature values tend to be associated with lower diagnosis values.
- **Correlation close to 0:** Indicates a weak linear relationship.

The features are sorted by their correlation with diagnosis and displayed in a horizontal bar chart. This helps identify which features have stronger or weaker linear associations with the target variable.

In [264]:
import plotly.express as px

numeric_cols = df_train.select_dtypes(
    include='number'
).columns

correlation = (
    df_train[numeric_cols]
    .corr()['diagnosis']
    .drop('diagnosis')
    .sort_values()
)

fig = px.bar(
    x=correlation.values,
    y=correlation.index,
    orientation='h',
    title='Feature Correlation with Diagnosis',
    labels={
        'x': 'Correlation',
        'y': 'Feature'
    }
)

fig.show()

### Correlation Heatmap of Medical Features

The following heatmap shows the pairwise correlations between the medical features in the training dataset.

The `patient_id` and `diagnosis` columns are excluded because the analysis focuses on relationships between the medical features themselves.

The correlation values range from **-1 to 1**:

- **1** indicates a strong positive linear relationship.
- **-1** indicates a strong negative linear relationship.
- **0** indicates little or no linear relationship.

The heatmap helps identify highly correlated features, which may provide similar information and could be relevant when selecting features for the machine learning model.

In [265]:
import plotly.express as px

medical_features = [
    col for col in df_train.select_dtypes(include='number').columns
    if col not in ['patient_id', 'diagnosis']
]

corr_matrix = df_train[medical_features].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    title='Correlation Heatmap of Medical Features',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1
)

fig.show()

### Stratified Group-Based Train-Validation Split

The dataset is divided into training and validation sets using `StratifiedGroupKFold`.

This approach is used to:

- Maintain a similar proportion of benign and malignant diagnoses in both sets through **stratification**.
- Ensure that observations from the same `patient_id` are kept within the same split through **grouping**.
- Reduce the risk of data leakage between the training and validation sets.
- Create **5 folds** using `n_splits=5`, with shuffling enabled for a more randomized split.

The first fold is selected as the validation set, while the remaining data is used for training. The diagnosis proportions are then compared between the two sets to verify that the class distribution is reasonably consistent.

In [266]:
from sklearn.model_selection import StratifiedGroupKFold

X = df_train.drop(columns=['diagnosis'])
y = df_train['diagnosis']
groups = df_train['patient_id']

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    sgkf.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()

y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nTrain diagnosis:")
print(y_train.value_counts(normalize=True))

print("\nValidation diagnosis:")
print(y_val.value_counts(normalize=True))

Train shape: (7064, 35)
Validation shape: (1763, 35)

Train diagnosis:
diagnosis
0    0.60957
1    0.39043
Name: proportion, dtype: float64

Validation diagnosis:
diagnosis
0    0.612592
1    0.387408
Name: proportion, dtype: float64


In [267]:
train_patients = set(X_train['patient_id'])
val_patients = set(X_val['patient_id'])

shared_patients = train_patients & val_patients

print("Patients shared between train and validation:",
      len(shared_patients))

Patients shared between train and validation: 0


### Data Preprocessing Pipeline

Before training the machine learning model, the features are prepared using a preprocessing pipeline.

The preprocessing steps include:

- **Removing `patient_id` and `exam_date`:** These columns are excluded because they are identifiers or date-related information that are not directly used as model features.
- **Categorical features:** `center` and `biopsy_followup_code` are treated as categorical variables.
- **Numerical features:** All remaining features are treated as numerical variables.
- **Missing numerical values:** Missing values are replaced with the median of each feature.
- **Feature scaling:** Numerical features are standardized using `StandardScaler`.
- **Missing categorical values:** Missing categorical values are replaced with the most frequent category.
- **One-hot encoding:** Categorical variables are converted into numerical representations using `OneHotEncoder`.

The `ColumnTransformer` combines the numerical and categorical preprocessing steps into a single preprocessing pipeline. This ensures that the same transformations can be consistently applied to both the training and validation data.

In [268]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Features to exclude
drop_features = ['patient_id', 'exam_date']

X_train_model = X_train.drop(columns=drop_features)
X_val_model = X_val.drop(columns=drop_features)

# Categorical features
categorical_features = [
    'center',
    'biopsy_followup_code'
]

# Numerical features
numeric_features = [
    col for col in X_train_model.columns
    if col not in categorical_features
]

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

### Logistic Regression Model

A Logistic Regression model is used as the classification algorithm to predict the `diagnosis` of each patient.

The model is combined with the preprocessing pipeline so that the data is automatically preprocessed before training.

The pipeline consists of:

1. **Preprocessing:** Missing values are handled, numerical features are standardized, and categorical features are one-hot encoded.
2. **Classification:** Logistic Regression is trained on the processed features.
3. **Model fitting:** The model is fitted using the training data and corresponding diagnosis labels.

The `max_iter=2000` parameter allows the model to perform enough iterations to reach convergence, while `random_state=42` ensures reproducible results.

In [269]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model.fit(X_train_model, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [270]:
y_val_pred = model.predict(X_val_model)
y_val_prob = model.predict_proba(X_val_model)[:, 1]

### Model Evaluation

The trained Logistic Regression model is evaluated on the validation dataset using several classification metrics.

The evaluation includes:

- **Classification Report:** Provides precision, recall, F1-score, and support for both benign and malignant classes.
- **ROC-AUC:** Measures the model's ability to distinguish between the two diagnosis classes across different classification thresholds. A higher value indicates better class separation.
- **Confusion Matrix:** Shows the number of correct and incorrect predictions for each diagnosis class.

The confusion matrix is also visualized as an interactive heatmap to make the model's predictions easier to interpret.

In [271]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import plotly.figure_factory as ff

# Classification report
print("Classification Report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=['Benign', 'Malignant']
    )
)

# ROC-AUC
roc_auc = roc_auc_score(y_val, y_val_prob)
print(f"ROC-AUC: {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred)

print("\nConfusion Matrix:")
print(cm)

# Interactive confusion matrix
fig = ff.create_annotated_heatmap(
    z=cm,
    x=['Predicted Benign', 'Predicted Malignant'],
    y=['Actual Benign', 'Actual Malignant'],
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title='Confusion Matrix - Logistic Regression',
    xaxis_title='Predicted Label',
    yaxis_title='Actual Label'
)

fig.show()

Classification Report:
              precision    recall  f1-score   support

      Benign       0.97      0.97      0.97      1080
   Malignant       0.95      0.96      0.96       683

    accuracy                           0.97      1763
   macro avg       0.96      0.97      0.96      1763
weighted avg       0.97      0.97      0.97      1763

ROC-AUC: 0.9795

Confusion Matrix:
[[1048   32]
 [  27  656]]


### ROC Curve and ROC-AUC

The Receiver Operating Characteristic (ROC) curve evaluates the ability of the Logistic Regression model to distinguish between benign and malignant cases across different classification thresholds.

The curve plots:

- **True Positive Rate (TPR):** The proportion of malignant cases correctly identified by the model.
- **False Positive Rate (FPR):** The proportion of benign cases incorrectly classified as malignant.

The **ROC-AUC** (Area Under the ROC Curve) summarizes the model's ability to distinguish between the two classes. A value closer to 1 indicates stronger class separation, while a value around 0.5 represents performance similar to a random classifier.

The dashed diagonal line represents the performance of a random classifier.

In [272]:
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go

# ROC curve
fpr, tpr, thresholds = roc_curve(
    y_val,
    y_val_prob
)

roc_auc = roc_auc_score(
    y_val,
    y_val_prob
)

fig = go.Figure()

# ROC curve
fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode='lines',
        name=f'ROC Curve (AUC = {roc_auc:.4f})'
    )
)

# Random classifier
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        name='Random Classifier',
        line=dict(dash='dash')
    )
)

fig.update_layout(
    title='ROC Curve - Logistic Regression',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    width=700,
    height=600
)

fig.show()

print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 0.9795


### Training vs Validation Performance

The model's performance is evaluated on both the training and validation datasets to assess how well it generalizes to unseen data.

The following metrics are compared:

- **Accuracy:** The proportion of all predictions that are correct.
- **Precision:** The proportion of predicted malignant cases that are actually malignant.
- **Recall:** The proportion of actual malignant cases that are correctly identified.
- **ROC-AUC:** Measures the model's ability to distinguish between benign and malignant cases across different classification thresholds.

Comparing the training and validation results helps identify potential **overfitting**. If the training performance is substantially higher than the validation performance, the model may be learning patterns that do not generalize well to unseen data.

In [274]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Predictions on training data
y_train_pred = model.predict(X_train_model)
y_train_prob = model.predict_proba(X_train_model)[:, 1]

# Predictions on validation data
y_val_pred = model.predict(X_val_model)
y_val_prob = model.predict_proba(X_val_model)[:, 1]

# Compare performance
results = {
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'ROC-AUC'
    ],
    'Train': [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_prob)
    ],
    'Validation': [
        accuracy_score(y_val, y_val_pred),
        precision_score(y_val, y_val_pred),
        recall_score(y_val, y_val_pred),
        roc_auc_score(y_val, y_val_prob)
    ]
}

results_df = pd.DataFrame(results)

print(results_df)

      Metric     Train  Validation
0   Accuracy  0.963477    0.966534
1  Precision  0.958211    0.953488
2     Recall  0.947788    0.960469
3    ROC-AUC  0.975415    0.979487


### Selecting an Optimal Classification Threshold

The default classification threshold of 0.5 is not always the most appropriate choice, especially when the cost of false positive and false negative predictions differs.

This analysis uses the **Precision-Recall curve** to identify a threshold that provides:

- **At least 90% precision**, meaning that at least 90% of cases predicted as malignant are actually malignant.
- The **highest possible recall** among the thresholds that meet the 90% precision requirement.

The threshold with the highest recall under this constraint is selected as the best threshold for the model.

In [275]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_prob
)

# Precision and recall corresponding to each threshold
precision_t = precision[:-1]
recall_t = recall[:-1]

# Keep only thresholds with precision >= 90%
valid = precision_t >= 0.90

valid_thresholds = thresholds[valid]
valid_precision = precision_t[valid]
valid_recall = recall_t[valid]

# Select the threshold with highest recall
best_idx = np.argmax(valid_recall)

best_threshold = valid_thresholds[best_idx]
best_precision = valid_precision[best_idx]
best_recall = valid_recall[best_idx]

print(f"Best Threshold: {best_threshold:.4f}")
print(f"Precision: {best_precision:.4f}")
print(f"Recall: {best_recall:.4f}")

Best Threshold: 0.0389
Precision: 0.9003
Recall: 0.9780


### Evaluation Using the Optimized Threshold

The selected threshold is now applied to the validation-set probabilities to generate the final class predictions.

The model is evaluated using:

- **Classification Report**: Provides precision, recall, F1-score, and support for both benign and malignant cases.
- **Confusion Matrix**: Shows the number of correctly and incorrectly classified cases.
- **Heatmap Visualization**: Provides a clear visual representation of the confusion matrix.

This evaluation shows how changing the classification threshold affects the model's ability to distinguish between benign and malignant cases.

In [276]:
from sklearn.metrics import confusion_matrix, classification_report
import plotly.figure_factory as ff

# Apply the new threshold
y_val_pred_threshold = (
    y_val_prob >= best_threshold
).astype(int)

# Classification report
print(
    classification_report(
        y_val,
        y_val_pred_threshold,
        target_names=['Benign', 'Malignant']
    )
)

# Confusion matrix
cm_threshold = confusion_matrix(
    y_val,
    y_val_pred_threshold
)

print("Confusion Matrix:")
print(cm_threshold)

# Plot
fig = ff.create_annotated_heatmap(
    z=cm_threshold,
    x=['Predicted Benign', 'Predicted Malignant'],
    y=['Actual Benign', 'Actual Malignant'],
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title=f'Confusion Matrix - Threshold {best_threshold:.4f}',
    xaxis_title='Predicted Label',
    yaxis_title='Actual Label'
)

fig.show()

              precision    recall  f1-score   support

      Benign       0.99      0.93      0.96      1080
   Malignant       0.90      0.98      0.94       683

    accuracy                           0.95      1763
   macro avg       0.94      0.95      0.95      1763
weighted avg       0.95      0.95      0.95      1763

Confusion Matrix:
[[1006   74]
 [  15  668]]


In [277]:
from sklearn.metrics import precision_score, recall_score

thresholds_to_test = [
    0.03,
    0.035,
    0.0389,
    0.04,
    0.045,
    0.05,
    0.06,
    0.07,
    0.08,
    0.10
]

threshold_results = []

for threshold in thresholds_to_test:
    y_pred = (y_val_prob >= threshold).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred)
    })

threshold_results = pd.DataFrame(threshold_results)

print(threshold_results.round(4))

   Threshold  Precision  Recall
0     0.0300     0.8063  0.9810
1     0.0350     0.8801  0.9780
2     0.0389     0.9015  0.9780
3     0.0400     0.9015  0.9780
4     0.0450     0.9064  0.9780
5     0.0500     0.9100  0.9766
6     0.0600     0.9162  0.9766
7     0.0700     0.9175  0.9766
8     0.0800     0.9187  0.9766
9     0.1000     0.9225  0.9766


In [278]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train_model, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [279]:
y_val_rf_pred = rf_model.predict(X_val_model)
y_val_rf_prob = rf_model.predict_proba(X_val_model)[:, 1]

In [280]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print("Classification Report:")
print(
    classification_report(
        y_val,
        y_val_rf_pred,
        target_names=['Benign', 'Malignant']
    )
)

rf_auc = roc_auc_score(
    y_val,
    y_val_rf_prob
)

print(f"ROC-AUC: {rf_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_rf_pred))

Classification Report:
              precision    recall  f1-score   support

      Benign       0.98      0.97      0.98      1080
   Malignant       0.96      0.98      0.97       683

    accuracy                           0.97      1763
   macro avg       0.97      0.97      0.97      1763
weighted avg       0.97      0.97      0.97      1763

ROC-AUC: 0.9891

Confusion Matrix:
[[1049   31]
 [  16  667]]


In [281]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision_rf, recall_rf, thresholds_rf = precision_recall_curve(
    y_val,
    y_val_rf_prob
)

valid_rf = precision_rf[:-1] >= 0.90

valid_thresholds_rf = thresholds_rf[valid_rf]
valid_precision_rf = precision_rf[:-1][valid_rf]
valid_recall_rf = recall_rf[:-1][valid_rf]

best_idx_rf = np.argmax(valid_recall_rf)

best_threshold_rf = valid_thresholds_rf[best_idx_rf]
best_precision_rf = valid_precision_rf[best_idx_rf]
best_recall_rf = valid_recall_rf[best_idx_rf]

print(f"Best Threshold: {best_threshold_rf:.4f}")
print(f"Precision: {best_precision_rf:.4f}")
print(f"Recall: {best_recall_rf:.4f}")

Best Threshold: 0.1833
Precision: 0.9000
Recall: 0.9883


In [282]:
from sklearn.metrics import brier_score_loss

brier_logistic = brier_score_loss(
    y_val,
    y_val_prob
)

brier_rf = brier_score_loss(
    y_val,
    y_val_rf_prob
)

print(f"Logistic Regression Brier Score: {brier_logistic:.4f}")
print(f"Random Forest Brier Score: {brier_rf:.4f}")

Logistic Regression Brier Score: 0.0299
Random Forest Brier Score: 0.0240


In [283]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np
import pandas as pd

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y, groups=groups), start=1
):
    X_cv_train = X.iloc[train_idx].drop(
        columns=['patient_id', 'exam_date']
    )

    X_cv_val = X.iloc[val_idx].drop(
        columns=['patient_id', 'exam_date']
    )

    y_cv_train = y.iloc[train_idx]
    y_cv_val = y.iloc[val_idx]

    rf_cv = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_cv.fit(X_cv_train, y_cv_train)

    y_cv_prob = rf_cv.predict_proba(X_cv_val)[:, 1]

    cv_results.append({
        'Fold': fold,
        'ROC-AUC': roc_auc_score(y_cv_val, y_cv_prob),
        'Brier Score': brier_score_loss(y_cv_val, y_cv_prob)
    })

cv_results = pd.DataFrame(cv_results)

print(cv_results.round(4))

print("\nMean:")
print(cv_results[['ROC-AUC', 'Brier Score']].mean().round(4))

print("\nStd:")
print(cv_results[['ROC-AUC', 'Brier Score']].std().round(4))

   Fold  ROC-AUC  Brier Score
0     1   0.9891       0.0240
1     2   0.9890       0.0263
2     3   0.9924       0.0270
3     4   0.9896       0.0239
4     5   0.9860       0.0282

Mean:
ROC-AUC        0.9892
Brier Score    0.0259
dtype: float64

Std:
ROC-AUC        0.0023
Brier Score    0.0019
dtype: float64


In [284]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
import numpy as np

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_prob = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y, groups=groups), start=1
):
    X_cv_train = X.iloc[train_idx].drop(
        columns=['patient_id', 'exam_date']
    )

    X_cv_val = X.iloc[val_idx].drop(
        columns=['patient_id', 'exam_date']
    )

    y_cv_train = y.iloc[train_idx]

    rf_oof = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_oof.fit(X_cv_train, y_cv_train)

    oof_prob[val_idx] = rf_oof.predict_proba(X_cv_val)[:, 1]


# Find threshold with precision >= 0.90
precision_oof, recall_oof, thresholds_oof = precision_recall_curve(
    y,
    oof_prob
)

valid_oof = precision_oof[:-1] >= 0.90

valid_thresholds = thresholds_oof[valid_oof]
valid_precision = precision_oof[:-1][valid_oof]
valid_recall = recall_oof[:-1][valid_oof]

best_idx = np.argmax(valid_recall)

best_threshold_oof = valid_thresholds[best_idx]
best_precision_oof = valid_precision[best_idx]
best_recall_oof = valid_recall[best_idx]

print(f"OOF Best Threshold: {best_threshold_oof:.4f}")
print(f"OOF Precision: {best_precision_oof:.4f}")
print(f"OOF Recall: {best_recall_oof:.4f}")

OOF Best Threshold: 0.1600
OOF Precision: 0.9005
OOF Recall: 0.9831


In [285]:
# Prepare full training data
X_full = df_train.drop(columns=['diagnosis'])
y_full = df_train['diagnosis']

X_full_model = X_full.drop(
    columns=['patient_id', 'exam_date']
)

# Final Random Forest
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# Train on all training data
final_model.fit(
    X_full_model,
    y_full
)

print("Final model trained successfully.")
print(f"Training samples: {len(X_full_model)}")

Final model trained successfully.
Training samples: 8827


In [286]:
# Prepare test data
X_test_model = df_test.drop(
    columns=['patient_id', 'exam_date']
)

# Predict probability of malignant (M)
test_prob = final_model.predict_proba(
    X_test_model
)[:, 1]

# Apply final OOF threshold
final_threshold = 0.1600

test_pred = (
    test_prob >= final_threshold
).astype(int)

# Convert 0/1 to B/M
predicted_label = np.where(
    test_pred == 1,
    'M',
    'B'
)

# Create submission
submission = pd.DataFrame({
    'patient_id': df_test['patient_id'],
    'predicted_label': predicted_label,
    'predicted_prob': test_prob
})

submission.head()

,patient_id,predicted_label,predicted_prob
0,912415,M,0.420000
1,910309,B,0.053333
2,911018,M,0.990000
3,911255,B,0.016667
4,910239,M,0.293333


In [287]:
print("Submission shape:", submission.shape)

print("\nLabels:")
print(submission['predicted_label'].value_counts())

print("\nMissing values:")
print(submission.isnull().sum())

print("\nDuplicate patient IDs:")
print(submission['patient_id'].duplicated().sum())

Submission shape: (2942, 3)

Labels:
predicted_label
B    1995
M     947
Name: count, dtype: int64

Missing values:
patient_id         0
predicted_label    0
predicted_prob     0
dtype: int64

Duplicate patient IDs:
58


In [288]:
duplicate_test_patients = (
    df_test[df_test['patient_id'].duplicated(keep=False)]
    .sort_values('patient_id')
)

print(duplicate_test_patients[
    ['patient_id', 'exam_date', 'center', 'biopsy_followup_code']
].head(20))

      patient_id  exam_date  center biopsy_followup_code
249       910088 2026-04-17  Site C              ROUTINE
1642      910088 2026-04-17  Site C              ROUTINE
338       910090 2025-11-30  Site A              ROUTINE
292       910090 2025-11-30  Site A              ROUTINE
480       910196 2025-08-16  Site B              ROUTINE
185       910196 2025-08-16  Site B              ROUTINE
2773      910202 2026-07-30  Site B              ROUTINE
1892      910202 2026-07-30  Site B              ROUTINE
36        910275 2025-04-02  Site B              ROUTINE
2651      910275 2025-04-02  Site B              ROUTINE
2181      910300 2025-09-21  Site B              ROUTINE
2894      910300 2025-09-21  Site B              ROUTINE
2468      910319 2025-01-19  Site B              ROUTINE
835       910319 2025-01-19  Site B              ROUTINE
590       910339 2026-09-05  Site B              ROUTINE
1125      910339 2026-09-05  Site B              ROUTINE
1834      910359 2026-06-13  Si

In [289]:
duplicate_rows_test = df_test[
    df_test['patient_id'].duplicated(keep=False)
].sort_values('patient_id')

print(
    duplicate_rows_test.groupby('patient_id').size().value_counts()
)

print(
    "Number of duplicated rows:",
    duplicate_rows_test.shape[0]
)

2    58
Name: count, dtype: int64
Number of duplicated rows: 116


In [290]:
submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv saved successfully.")

submission.csv saved successfully.
